# 07 — Fraud labeling (Part D 2019–2023)

Same labeling as Notebook 04 (Part B), applied to Part D. Nothing is rebuilt: the cumulative LEIE table from Notebook 04 is reused, so both programs share one label definition.

Input:
- `processed/part_d/part_d_pnd_npi_year_2019_2023.parquet` — Part D by Provider and Drug, aggregated to NPI + year (5,148,303 × 33)
- `processed/part_b/labeled/leie_cumulative_2019_2023.parquet` — from Notebook 04

Output: `processed/part_d/labeled/part_d_A_labeled_2019_2023.parquet` (same rows + label columns)

Label columns:
- `fraud` — the paper's rule (identical to Part B): `year < last_excl_year`.
- `fraud_h2`, `fraud_h3` — ⚠️ not in the paper. Same rule, but a row counts as fraud only if the exclusion date is at most H years after the claim year. Controls right-censoring: with an unbounded look-ahead, 2019 rows get ~7 years to be caught and 2023 rows ~3. These are for the sensitivity check; `fraud` stays the replication label.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
PD_DIR   = DATA_DIR / "processed" / "part_d"
OUT_DIR  = PD_DIR / "labeled"; OUT_DIR.mkdir(parents=True, exist_ok=True)

AGG_D    = PD_DIR / "part_d_pnd_npi_year_2019_2023.parquet"
LEIE_CUM = DATA_DIR / "processed/part_b/labeled/leie_cumulative_2019_2023.parquet"
PB_LAB   = DATA_DIR / "processed/part_b/labeled/part_b_labeled_2019_2023.parquet"
OUT      = OUT_DIR / "part_d_A_labeled_2019_2023.parquet"

LABEL_RULE = "<"       # must match Notebook 04
HORIZONS   = [2, 3]    # years; for fraud_h2, fraud_h3
for p in (AGG_D, LEIE_CUM, PB_LAB): print(p.exists(), p)

True /scratch/sm3664/medicare-fraud/processed/part_d/part_d_pnd_npi_year_2019_2023.parquet
True /scratch/sm3664/medicare-fraud/processed/part_b/labeled/leie_cumulative_2019_2023.parquet
True /scratch/sm3664/medicare-fraud/processed/part_b/labeled/part_b_labeled_2019_2023.parquet


## Step 1 — Sanity checks on the aggregated file

Checks the things the paper's Part D preparation depends on:
- one row per NPI + year, 30 stat columns (5 fields × 6 stats)
- no negative counts or costs
- std = 0 when min = max (single-row providers)
- `total_benes`: CMS blanks counts below 11. The paper imputes 5. If imputation was done, values 1–10 other than 5 should not appear.
- provider type spelling variants (paper merged some Part D variants)

In [2]:
df = pd.read_parquet(AGG_D)
print(df.shape)

KEYS  = ["npi", "year", "provider_type"]
BASES = ["total_claims", "total_30day_fills", "total_day_supply", "total_drug_cost", "total_benes"]
STATS = ["min", "max", "median", "mean", "sum", "std"]
AGG   = [f"{b}_{s}" for b in BASES for s in STATS]

assert set(KEYS + AGG) == set(df.columns), set(df.columns) ^ set(KEYS + AGG)
assert not df.duplicated(["npi", "year"]).any(), "duplicate NPI-year"
assert df["npi"].between(1_000_000_000, 9_999_999_999).all(), "NPI not 10 digits"
assert df[AGG].notna().all().all(), "NaN in stats"
print("rows per year:\n", df.groupby("year").size())

neg = (df[AGG] < 0).sum(); print("negative values:", neg[neg > 0].to_dict() or "none")

for b in BASES:
    single = df[f"{b}_min"] == df[f"{b}_max"]
    bad = (single & (df[f"{b}_std"] != 0)).sum()
    print(f"{b:>18}: min==max rows {single.mean():.1%}, of which std != 0: {bad}")

(5148303, 33)
rows per year:
 year
2019     985532
2020     983635
2021    1017412
2022    1057564
2023    1104160
dtype: int64
negative values: none
      total_claims: min==max rows 16.7%, of which std != 0: 0
 total_30day_fills: min==max rows 16.4%, of which std != 0: 0
  total_day_supply: min==max rows 16.1%, of which std != 0: 0
   total_drug_cost: min==max rows 15.9%, of which std != 0: 0
       total_benes: min==max rows 25.4%, of which std != 0: 0


In [3]:
low = df.loc[df.total_benes_min < 11, "total_benes_min"]
print("total_benes_min values below 11:\n", low.value_counts().sort_index())
print("→ only 5 appears = suppressed counts were imputed with 5 (paper). "
      "Other values 1–10 = not imputed; check how blanks were handled.")

total_benes_min values below 11:
 total_benes_min
5.0    3965481
Name: count, dtype: int64
→ only 5 appears = suppressed counts were imputed with 5 (paper). Other values 1–10 = not imputed; check how blanks were handled.


In [4]:
pt = df["provider_type"]
print("provider types:", pt.nunique())
norm = pt.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
variants = pt.groupby(norm).unique()
variants = variants[variants.str.len() > 1]
print("spelling variants (same letters, different text):", len(variants))
print(variants.head(30).to_string())

provider types: 210
spelling variants (same letters, different text): 0
Series([], )


## Step 2 — Join the cumulative LEIE table and label

Left join on NPI so every Part D row is kept. The LEIE table holds every fraud-rule NPI in the LEIE, not only those found in Part B.

In [5]:
cum = pd.read_parquet(LEIE_CUM)
assert cum.NPI.is_unique
cum["excl_year"] = cum.EXCLDATE.dt.year
print("LEIE fraud NPIs:", len(cum)); print(cum.EXCLTYPE.value_counts())

n0 = len(df)
df = df.merge(cum[["NPI", "EXCLTYPE", "excl_year", "last_excl_year"]],
              how="left", left_on="npi", right_on="NPI").drop(columns="NPI")
assert len(df) == n0, "row count changed"

cmp = df.year < df.last_excl_year if LABEL_RULE == "<" else df.year <= df.last_excl_year
df["fraud"] = cmp.fillna(False).astype("int8")
for h in HORIZONS:
    within = (df.excl_year - df.year) <= h
    df[f"fraud_h{h}"] = (df.fraud.astype(bool) & within.fillna(False)).astype("int8")

LABELS = ["fraud"] + [f"fraud_h{h}" for h in HORIZONS]
print("NPIs matched in Part D:", df.loc[df.last_excl_year.notna(), "npi"].nunique())

LEIE fraud NPIs: 7970
EXCLTYPE
1128a1    3767
1128b4    2805
1128a3     756
1128a2     424
1128b7     218
Name: count, dtype: int64
NPIs matched in Part D: 1136


## Step 3 — Label summary

- Fraud rows, NPIs and rate per year for each label. With `fraud`, the rate should fall toward 2023 (right-censoring, as in Part B). With `fraud_h2` it should be flatter; if it is, censoring explains the decline.
- Exclusion types among fraud NPIs (Part B: ~35% were 1128(b)(4)).

In [6]:
for lab in LABELS:
    s = df.groupby("year")[lab].agg(rows="size", fraud="sum", rate="mean")
    s["rate"] = (100 * s.rate).round(4).astype(str) + "%"
    n_npi = df.loc[df[lab] == 1, "npi"].nunique()
    print(f"\n{lab}: {int(df[lab].sum())} rows, {n_npi} NPIs, "
          f"{100 * df[lab].mean():.4f}% overall\n{s}")

fraud_npis = df.loc[df.fraud == 1].drop_duplicates("npi")
print("\nExclusion type of fraud NPIs:\n", fraud_npis.EXCLTYPE.value_counts(normalize=True).round(3))


fraud: 3274 rows, 1065 NPIs, 0.0636% overall
         rows  fraud     rate
year                         
2019   985532    991  0.1006%
2020   983635    790  0.0803%
2021  1017412    648  0.0637%
2022  1057564    497   0.047%
2023  1104160    348  0.0315%

fraud_h2: 958 rows, 601 NPIs, 0.0186% overall
         rows  fraud     rate
year                         
2019   985532    154  0.0156%
2020   983635    184  0.0187%
2021  1017412    189  0.0186%
2022  1057564    219  0.0207%
2023  1104160    212  0.0192%

fraud_h3: 1653 rows, 877 NPIs, 0.0321% overall
         rows  fraud     rate
year                         
2019   985532    306   0.031%
2020   983635    299  0.0304%
2021  1017412    357  0.0351%
2022  1057564    343  0.0324%
2023  1104160    348  0.0315%

Exclusion type of fraud NPIs:
 EXCLTYPE
1128b4    0.420
1128a1    0.412
1128a3    0.100
1128a2    0.048
1128b7    0.021
Name: proportion, dtype: float64


## Step 4 — Overlap with Part B

Needed for the combined B + D dataset: how many fraud providers appear in both programs, and how many in only one.

In [7]:
pb = pd.read_parquet(PB_LAB, columns=["Rndrng_NPI", "fraud"])
b_all, b_fr = set(pb.Rndrng_NPI), set(pb.loc[pb.fraud == 1, "Rndrng_NPI"])
d_all, d_fr = set(df.npi), set(df.loc[df.fraud == 1, "npi"])
print(f"All NPIs   — B only {len(b_all - d_all):,} | both {len(b_all & d_all):,} | D only {len(d_all - b_all):,}")
print(f"Fraud NPIs — B only {len(b_fr - d_fr):,} | both {len(b_fr & d_fr):,} | D only {len(d_fr - b_fr):,}")
del pb

All NPIs   — B only 552,568 | both 941,210 | D only 458,074
Fraud NPIs — B only 226 | both 704 | D only 361


In [8]:
df.drop(columns=["EXCLTYPE", "excl_year", "last_excl_year"]).to_parquet(OUT, index=False)
print(OUT, df.shape)

/scratch/sm3664/medicare-fraud/processed/part_d/labeled/part_d_A_labeled_2019_2023.parquet (5148303, 39)
